# 02 — Exploratory data analysis

Reads **only** the outputs of `01_data_engineering_pipeline.ipynb` (never
reconstructs or modifies them). Two kinds of statements are kept separate here:

- **descriptive findings** about the corpus and the constructed datasets;
- **linkage-quality evidence**, which lives in `03_analysis.ipynb` — in
  particular, a healthy-looking composite-score distribution among selected
  links is NOT independent validation, because the same score selected them.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from boamp.config import load_config
from boamp.reporting.figures import setup_style, save_figure

cfg = load_config(PROJECT_ROOT)
setup_style()
D1, D2, D3 = cfg.paths.processed_boamp_only, cfg.paths.processed_enriched, cfg.paths.processed_comparison

DATE_COLS_SRC = ["publication_date", "start_date", "estimated_end_date", "study_end_date"]
STR_COLS = {c: str for c in ["notice_id", "buyer_key", "buyer_key_type", "cpv_clean",
                             "cpv_division", "cpv_group", "cpv_class", "cpv_category"]}

common = pd.read_csv(cfg.paths.interim_common_prepared, low_memory=False, dtype=str,
                     usecols=["notice_id", "notice_type_normalized", "dateparution",
                              "buyer_key_type", "cpv_division", "cpv_missing", "objet_clean",
                              "is_digital_scope", "duration_quality_flag", "schema_family"])
common["publication_date"] = pd.to_datetime(common["dateparution"], errors="coerce", utc=True).dt.tz_localize(None)
sources = pd.read_csv(D1 / "boamp_only_sources.csv", dtype=STR_COLS, parse_dates=DATE_COLS_SRC)
l2_sources = pd.read_csv(D2 / "enriched_sources.csv", dtype=STR_COLS, parse_dates=DATE_COLS_SRC)
l1_pairs = pd.read_csv(D1 / "boamp_only_candidate_pairs.csv")
l2_pairs = pd.read_csv(D2 / "enriched_candidate_pairs.csv")
l1_links = pd.read_csv(D1 / "boamp_only_links_balanced.csv")
l2_links = pd.read_csv(D2 / "enriched_links_balanced.csv")
l1_surv = pd.read_csv(D1 / "boamp_only_survival.csv", parse_dates=["publication_date"])
l2_surv = pd.read_csv(D2 / "enriched_survival.csv", parse_dates=["publication_date"])
link_cmp = pd.read_csv(D3 / "layer_link_comparison.csv")
print(f"common {len(common):,} | sources {len(sources):,} | L1 pairs {len(l1_pairs):,} | "
      f"L2 pairs {len(l2_pairs):,} | L1 links {len(l1_links)} | L2 links {len(l2_links)}")

common 84,623 | sources 3,380 | L1 pairs 10,862 | L2 pairs 13,524 | L1 links 1003 | L2 links 1188


## 1. Corpus over time and notice types

The full cleaned corpus (all sectors) vs the digital-scope source population.

In [2]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
by_year = common.groupby([common["publication_date"].dt.year, "notice_type_normalized"]).size().unstack(fill_value=0)
by_year.plot(kind="bar", stacked=True, ax=axes[0], width=0.85)
axes[0].set_title("All cleaned notices by year and type")
axes[0].set_xlabel("")
sources.groupby(sources["publication_date"].dt.year).size().plot(kind="bar", ax=axes[1], width=0.85)
axes[1].set_title("Eligible digital-scope sources by year (n=3,159)")
axes[1].set_xlabel("")
save_figure(fig, "eda_corpus_over_time", cfg)
plt.show()

## 2. Missingness and data quality

Duration quality is the corpus's weakest field: only ~12% of sources carry an
observed duration; the rest are imputed with the CPV-division median. Everything
downstream that depends on the estimated end date inherits this.

In [3]:
dq = pd.DataFrame({
    "all_notices": {
        "cpv_missing": (common["cpv_missing"] == "True").mean(),
        "text_missing": common["objet_clean"].isna().mean(),
        "buyer_key_missing": (common["buyer_key_type"] == "MISSING").mean(),
    },
    "sources": {
        "cpv_missing": sources["cpv_clean"].isna().mean(),
        "text_missing": sources["objet_clean"].isna().mean(),
        "buyer_key_missing": (sources["buyer_key_type"] == "MISSING").mean(),
    },
}).T
display(dq.style.format("{:.2%}"))
print("duration quality among sources:")
print(sources["duration_quality_flag"].value_counts(normalize=True).map("{:.1%}".format))

,cpv_missing,text_missing,buyer_key_missing
all_notices,17.06%,0.00%,0.00%
sources,13.93%,0.00%,0.00%


duration quality among sources:
duration_quality_flag
IMPUTED_MEDIAN_BY_CPV_DIVISION    82.7%
OBSERVED                          17.3%
Name: proportion, dtype: object


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
obs = sources[~sources["dur_was_imputed"].astype(str).str.lower().eq("true")]["declared_duration_months"].astype(float)
imp = sources[sources["dur_was_imputed"].astype(str).str.lower().eq("true")]["declared_duration_months"].astype(float)
axes[0].hist([obs, imp], bins=30, stacked=True, label=[f"observed (n={len(obs)})", f"imputed (n={len(imp)})"])
axes[0].legend(); axes[0].set_title("Declared duration (months): observed vs imputed")
cpv_cov = sources.assign(year=sources["publication_date"].dt.year).groupby("year")["cpv_clean"]     .apply(lambda s: s.notna().mean())
cpv_cov.plot(ax=axes[1], marker="o")
axes[1].set_ylim(0, 1); axes[1].set_title("CPV coverage among sources by year")
save_figure(fig, "eda_duration_and_cpv_quality", cfg)
plt.show()

## 3. Buyer identity: fragmentation and enrichment coverage (L1 vs L2)

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
kt = pd.DataFrame({
    "Layer 1 key type": sources["buyer_key_type"].value_counts(),
    "Layer 2 identity source": l2_sources["buyer_identity_source"].value_counts(),
})
sources["buyer_key_type"].value_counts().plot(kind="barh", ax=axes[0])
axes[0].set_title("Layer 1: buyer_key_type (sources)")
l2_sources["buyer_identity_source"].value_counts().plot(kind="barh", ax=axes[1])
axes[1].set_title("Layer 2: buyer_identity_source (sources)")
save_figure(fig, "eda_buyer_identity_l1_l2", cfg)
plt.show()
n1 = sources["buyer_key"].nunique()
n2 = l2_sources["buyer_key_l2"].nunique()
print(f"unique buyers: L1 {n1} -> L2 {n2}  (merge effect: {n1 - n2:+d})")
print(f"L2 SIREN coverage: {l2_sources['buyer_siren_l2'].notna().mean():.1%} "
      f"(native effective: {l2_sources['buyer_siren_boamp_effective'].notna().mean():.1%})")

unique buyers: L1 819 -> L2 808  (merge effect: +11)
L2 SIREN coverage: 63.5% (native effective: 25.9%)


In [6]:
top = sources.groupby("buyer_key").size().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8, 4.5))
top.sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 15 buyers by number of eligible sources (buyer concentration)")
save_figure(fig, "eda_buyer_concentration", cfg)
plt.show()

## 4. Candidate generation and score distributions

Distributions below are **descriptive**. The rank-1 score distribution around the
frozen thresholds shows how much mass sits near the decision boundary (the
threshold-sensitivity analysis in 03 quantifies the resulting flip risk).

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
cand_counts = l1_pairs.groupby("source_notice_id").size()
axes[0].hist(cand_counts, bins=range(1, 32))
axes[0].set_title(f"L1 candidates per source (median={cand_counts.median():.0f}, capped at 30)")
r1 = l1_pairs[l1_pairs["candidate_rank"] == 1]
axes[1].hist(r1["composite_score"], bins=50)
for name, color in [("broad", "tab:green"), ("balanced", "tab:orange"), ("strict", "tab:red")]:
    axes[1].axvline(getattr(cfg.pipeline.thresholds, name), color=color, ls="--", label=name)
axes[1].legend(); axes[1].set_title("L1 rank-1 composite score with frozen thresholds")
save_figure(fig, "eda_candidates_and_scores", cfg)
plt.show()

In [8]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, col, title in [(axes[0], "s_text", "text similarity"),
                       (axes[1], "s_time", "temporal score"),
                       (axes[2], "top1_top2_margin", "top1-top2 margin (rank-1)")]:
    data = r1[col].dropna() if col != "top1_top2_margin" else r1["top1_top2_margin"].dropna()
    ax.hist(data, bins=40)
    ax.set_title(f"L1 rank-1 {title}")
axes[2].axvline(cfg.pipeline.confidence_tiers.potential_margin_max, color="tab:red", ls="--",
                label="POTENTIAL cutoff")
axes[2].legend()
save_figure(fig, "eda_component_scores", cfg)
plt.show()

## 5. Linked vs unlinked: selection profile and censoring

Linked sources are a selected subsample — differences here flag potential
selection bias for the survival analysis (formally examined in 03).

In [9]:
for label, surv in [("Layer 1", l1_surv), ("Layer 2", l2_surv)]:
    ev = surv["event"] == 1
    print(f"{label}: events {ev.sum()} ({ev.mean():.1%}), censored {(~ev).sum()} ({(~ev).mean():.1%})")
prof = l1_surv.assign(year=l1_surv["publication_date"].dt.year)
sel = prof.groupby("event").agg(
    n=("notice_id", "count"),
    median_duration=("declared_duration_months", "median"),
    imputed_rate=("dur_was_imputed", "mean"),
    median_year=("year", "median"),
)
display(sel)

Layer 1: events 1003 (29.7%), censored 2377 (70.3%)
Layer 2: events 1188 (35.1%), censored 2192 (64.9%)


,n,median_duration,imputed_rate,median_year
event,,,,
0,2377,6.0,0.810686,2021.0
1,1003,6.0,0.865404,2020.0


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for label, surv, ax in [("Layer 1 (boamp_only)", l1_surv, axes[0]), ("Layer 2 (enriched)", l2_surv, axes[1])]:
    yr = surv.assign(year=surv["publication_date"].dt.year).groupby("year")["event"].agg(["mean", "count"])
    ax.bar(yr.index, yr["mean"])
    ax.set_title(f"{label}: event rate by publication year")
    ax.set_ylim(0, 0.65)
save_figure(fig, "eda_event_rate_by_year", cfg)
plt.show()

In [11]:
cpv_rates = pd.DataFrame({
    "L1": l1_surv.groupby("cpv_division")["event"].mean(),
    "L2": l2_surv.groupby("cpv_division")["event"].mean(),
    "n": l1_surv.groupby("cpv_division").size(),
}).query("n >= 30").sort_values("n", ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
cpv_rates[["L1", "L2"]].plot(kind="bar", ax=ax, width=0.8)
ax.set_title("Event rate by CPV division (divisions with n>=30), L1 vs L2")
save_figure(fig, "eda_event_rate_by_cpv", cfg)
plt.show()

## 6. Layer comparison at a glance

Where the layers disagree — the added/changed links are exactly the population
whose quality decides whether enrichment helps (analyzed in depth in 03).

In [12]:
status = link_cmp["link_status"].value_counts()
display(status.to_frame("n_sources"))
added_ids = link_cmp.loc[link_cmp["link_status"] == "ADDED_BY_ENRICHMENT", "source_notice_id"]
added_links = l2_links[l2_links["source_notice_id"].isin(set(added_ids))]
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist([l1_links["s_text"].dropna(), added_links["s_text"].dropna()], bins=30, density=True,
        label=[f"L1 links (n={len(l1_links)})", f"links added by enrichment (n={len(added_links)})"])
ax.legend(); ax.set_title("Text similarity: L1 links vs enrichment-added links (descriptive)")
save_figure(fig, "eda_added_links_text_similarity", cfg)
plt.show()
print(f"median s_text: L1 {l1_links['s_text'].median():.3f} vs added {added_links['s_text'].median():.3f}")
print("NOTE: weaker text similarity among added links is a quality WARNING, examined in 03.")

,n_sources
link_status,
UNLINKED_BOTH,2188
SAME_LINK,962
ADDED_BY_ENRICHMENT,189
CHANGED_CANDIDATE,37
REMOVED_BY_ENRICHMENT,4


median s_text: L1 0.207 vs added 0.087
NOTE: weaker text similarity among added links is a quality WARNING, examined in 03.


In [13]:
print("EDA complete. Figures saved to", cfg.paths.reports_figures)

EDA complete. Figures saved to /Users/macbookair/Documents/ENSAE/2A(senghak)/survival-analysis/reports/figures
